# FPGrowth and Collaborative Filtering

## Introduction

### 1. Overall idea of the hybrid model

The hybrid pipeline is:

1. **Stage 1 (Pattern mining)**  
   - From historical transaction / basket data (e.g. user–item interactions), mine **frequent itemsets** (and optionally association rules) using **FPGrowth**.
   - Use these patterns to:
     - pre‑filter candidate items,
     - enrich the user–item matrix, or
     - generate "warm‑up" recommendations before CF.

2. **Stage 2 (Personalization)**  
   - Feed the refined input (e.g. filtered items, or boosted weights from FP‑patterns) into a **Collaborative Filtering** model (e.g. user‑based, item‑based, or matrix‑factorization‑based).
   - The CF model then outputs **personalized recommendations** for each user.

This two‑stage design helps overcome **sparsity and cold‑start** issues by leveraging association patterns first, then fine‑tuning with user‑behavior similarity. [pdfs.semanticscholar](https://pdfs.semanticscholar.org/6259/f01d784a0b3c7a4a614eb8e75e3ab086bced.pdf)


### 2. Stage 1: FPGrowth
#### Input notation
Let:
- $D = \{T_1, T_2, \dots, T_n\}$ be a **transaction database**,  
  where $T_i \subseteq I$ is a transaction (basket) and $I = \{i_1, i_2, \dots, i_m\}$ is the set of all items. [philippe-fournier-viger](https://www.philippe-fournier-viger.com/spmf/FPGrowth.php)
- $\text{minsup} \in (0,1]$ the **minimum support threshold**.
- $\text{supp}(X)$ the **support** of itemset $X$:  
  $$
  \text{supp}(X) = \frac{\#\{T \in D \mid X \subseteq T\}}{|D|}
  $$
  i.e., the fraction of transactions containing $X$. [philippe-fournier-viger](https://www.philippe-fournier-viger.com/spmf/FPGrowth.php)
  
  A **frequent itemset** $X$ is one where:
$$
\text{supp}(X) \geq \text{minsup}
$$

#### FP‑Tree and mining (high‑level math)
The FP‑Growth algorithm proceeds in two main steps:
1. **Build the FP‑Tree**  
   - Scan the database once to compute $f(j)$, the frequency of item $j$.
   - Remove items with $f(j) < \text{minsup}$ (infrequent).
   - Order the remaining items by frequency (descending).
   - For each transaction $T_i$:
     - keep only the frequent items,
     - sort them by frequency order,
     - insert this ordered itemset into the **FP‑Tree** (a compressed prefix tree with counts). [geeksforgeeks](https://www.geeksforgeeks.org/machine-learning/frequent-pattern-growth-algorithm/)

2. **Mine frequent itemsets via conditional FP‑Trees**  
   - For each frequent item $z$, build a **conditional pattern base**:
     $$
     \text{CPB}(z) = \bigcup_{\text{all paths ending with } z} \text{prefix paths}
     $$
   - From $\text{CPB}(z)$, construct a **conditional FP‑Tree** for item $z$, and recursively mine patterns of the form:
   $$
     X \cup \{z\}
   $$
     
     where $X$ is a frequent itemset in the conditional tree. [geeksforgeeks](https://www.geeksforgeeks.org/machine-learning/frequent-pattern-growth-algorithm/)

#### Optional: association rules
After getting frequent itemsets, you can derive **association rules** $X \rightarrow Y$ where $X \cap Y = \emptyset$.
- **Support** of rule $X \rightarrow Y$:
  $$
  \text{supp}(X \rightarrow Y) = \text{supp}(X \cup Y)
  $$

- **Confidence** of rule $X \rightarrow Y$:
  $$
  \text{conf}(X \rightarrow Y) = \frac{\text{supp}(X \cup Y)}{\text{supp}(X)}
  $$

In your hybrid model, you might keep only rules with $\text{supp} \geq \text{minsup}$ and $\text{conf} \geq \text{minconf}$. [scaler](https://www.scaler.com/topics/data-mining-tutorial/fp-growth-in-data-mining/)

#### How you use FP‑results in Stage 1 output
From Stage 1 you can output:
- A set of frequent itemsets $\mathcal{F} = \{X \mid \text{supp}(X) \geq \text{minsup}\}$.
- Or a set of strong association rules $\mathcal{R} = \{X \rightarrow Y \mid \text{supp} \geq \text{minsup}, \text{conf} \geq \text{minconf}\}$.

You can then:
- **Boost item co‑occurrence weights** in the user–item matrix (e.g., add small weights proportional to support or confidence).
- Or **pre‑filter candidates**: for user $u$, only consider items $i$ that appear in frequent patterns involving items $u$ has interacted with. [jiita](http://jiita.org/jiita/vol7/JIITA_Vol7_No1_p.654-665.pdf)

***

### 3. Stage 2: Collaborative Filtering (math & equations)
Let:
- $U = \{u_1, u_2, \dots, u_n\}$ be the set of **users**.
- $I = \{i_1, i_2, \dots, i_m\}$ be the set of **items**.
- $R \in \mathbb{R}^{n \times m}$ be the **rating / interaction matrix**, where $r_{u i}$ is the rating (or implicit feedback, e.g. 1 = purchased, 0 = not) that user $u$ gave to item $i$. [ibm](https://www.ibm.com/think/topics/collaborative-filtering)

#### Option A: User‑based or item‑based CF (neighborhood methods)
##### Similarity (cosine as example)
For **user‑based CF**, similarity between users $u$ and $v$ (over items they both rated):
$$
\text{sim}(u, v) = \frac{
\sum_{i \in I_{uv}} (r_{u i} - \bar{r}_u)(r_{v i} - \bar{r}_v)
}{
\sqrt{\sum_{i \in I_{uv}} (r_{u i} - \bar{r}_u)^2}
\sqrt{\sum_{i \in I_{uv}} (r_{v i} - \bar{r}_v)^2}
}
$$
where:
- $I_{uv}$ = items rated by both $u$ and $v$,
- $\bar{r}_u$ = average rating of user $u$. [cs.princeton](https://www.cs.princeton.edu/courses/archive/spring17/cos435/Notes/search_refine_rec_topostPart2.pdf)

In **item‑based CF**, same formula applies to items $i$ and $j$, with similarity over users who rated both.

##### Prediction (user‑based example)
To predict rating $\hat{r}_{u i}$ for user $u$ and item $i$:
$$
\hat{r}_{u i} = \bar{r}_u +
\frac{
\sum_{v \in N_u(i)} \text{sim}(u, v) (r_{v i} - \bar{r}_v)
}{
\sum_{v \in N_u(i)} |\text{sim}(u, v)|
}
$$
where $N_u(i)$ is the set of “neighbor” users of $u$ who have rated item $i$. [ibm](https://www.ibm.com/think/topics/collaborative-filtering)

You can use FP‑Growth results to:
- **restrict the candidate set** to items related via strong association rules to $u$’s history,
- or to **weight the similarity** by pattern strength (e.g., higher similarity if users share frequent itemsets).

#### Option B: Matrix factorization CF (e.g., SVD / ALS)
Represent users and items in latent space:
- Each user $u$ → vector $\mathbf{p}_u \in \mathbb{R}^k$.
- Each item $i$ → vector $\mathbf{q}_i \in \mathbb{R}^k$.

Then predicted rating:
$$
\hat{r}_{u i} = \mathbf{p}_u^\top \mathbf{q}_i
$$

Optimization (minimize reconstruction loss + regularization):
$$
\min_{\mathbf{P}, \mathbf{Q}} \sum_{(u,i) \in \mathcal{O}} (r_{u i} - \mathbf{p}_u^\top \mathbf{q}_i)^2
+ \lambda \left(
\|\mathbf{P}\|_{F}^2 + \|\mathbf{Q}\|_{F}^2
\right)
$$
where:
- $\mathcal{O}$ = observed ratings,
- $\|\cdot\|_F$ is Frobenius norm,
- $\lambda$ controls regularization. [cs.princeton](https://www.cs.princeton.edu/courses/archive/spring17/cos435/Notes/search_refine_rec_topostPart2.pdf)

You can **incorporate FP‑Growth patterns** into MF‑like CF by:
- Using **support/confidence values** to **increase weights** of related item pairs in the loss,
- Or **initializing** $\mathbf{q}_i$ vectors such that items appearing together in frequent patterns are closer in the embedding space.

***

### 4. How the two stages connect (hybrid logic)
A concrete pipeline:
1. **Stage 1 (FPGrowth)**  
   - Input: transaction data $D$ (e.g., orders, baskets).
   - Output: frequent itemsets $\mathcal{F}$ or association rules $\mathcal{R}$.

2. **Refine/weight the user–item matrix using FP‑results**  
   - For each user $u$, identify all items $I_u$ they have interacted with.
   - For each association rule $X \rightarrow Y$ where $X \subseteq I_u$ and $Y \not\subseteq I_u$:
     - Add a **weight** $w_{u y} = \text{conf}(X \rightarrow Y)$ or a function of $\text{supp}(X \cup Y)$ to item $y$ for user $u$.
   - This effectively creates a **pre‑filtered and boosted interaction matrix** $R'$.

3. **Stage 2 (CF)**  
   - Feed $R'$ (or the raw $R$ with FP‑enhanced weights) into:
     - a **user‑/item‑based CF** model, or
     - a **matrix‑factorization** model.
   - CF outputs predicted scores $\hat{r}_{u i}$ for un‑interacted items $i$.

4. **Final recommendation**  
   - For user $u$, rank items $i$ by $\hat{r}_{u i}$,
   - Output top‑$K$ items as recommendations.

This **two‑stage hybrid** leverages **global patterns from FPGrowth** first, then **personalizes** with CF, improving robustness to sparsity and cold start. [ijcrt](https://www.ijcrt.org/papers/IJCRT2603275.pdf)

***

# Module Loading

In [8]:
import gdown
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pylab import rcParams
import pandas as pd
import numpy as np
from google.colab import drive
from warnings import filterwarnings

drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [10]:
# Modern Professional Color Palette
modern_colors = [
    "#1f77b4",   # Vibrant Blue (Primary)
    "#ff7f0e",   # Bright Orange (Accent/Comparison)
    "#2ca02c",   # Fresh Green (Success/Positive)
    "#d62728",   # Soft Red (Alert/Warning)
    "#9467bd",   # Elegant Purple
    "#8c564b",   # Warm Brown
    "#e377c2",   # Pink
    "#7f7f7f",   # Neutral Gray
    "#bcbd22",   # Olive/Yellow-Green
    "#17becf"    # Cyan/Teal
]

# 2. Main Style Dictionary
modern_light_style = {
    # Background - Clean and bright
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#f8f9fa",
    "savefig.facecolor": "#ffffff",

    # Grid - Very subtle and non-distracting
    "axes.grid": True,
    "grid.color": "#e6e8eb",
    "grid.linestyle": "--",
    "grid.linewidth": 0.8,
    "axes.grid.which": "both",

    # Typography
    "text.color": "#1f2937",
    "axes.labelcolor": "#1f2937",
    "xtick.color": "#374151",
    "ytick.color": "#374151",
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.titlepad": 18,
    "font.size": 12,
    "font.family": "sans-serif", # You can change to 'Arial', 'Helvetica', etc.

    # Spines - Clean and minimal
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": True,
    "axes.spines.bottom": True,
    "axes.edgecolor": "#4b5563",
    "axes.linewidth": 1.2,

    # Lines and markers
    "axes.prop_cycle": plt.cycler(color=modern_colors),
    "lines.linewidth": 2.5,
    "lines.markersize": 7,
    "lines.markeredgewidth": 0.8,

    # Patches (bars, areas, etc.)
    "patch.edgecolor": "#ffffff",
    "patch.linewidth": 0.8,

    # Legend
    "legend.frameon": False,
    "legend.loc": "best",
    "legend.fontsize": 11,
}

plt.style.use('default')
sns.set_theme(style="whitegrid", rc=modern_light_style)
plt.rcParams.update(modern_light_style)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=modern_colors)

In [3]:
def CSV_gdrive(file_id:str,
               output_filename: str= None,
               show_head : bool= True,
              ) -> pd.DataFrame:
    if output_filename is None:
        output_filename = 'downloaded_file.csv'
    gdrive_url = f'https://drive.google.com/uc?id={file_id}'
    df = pd.DataFrame()
    try:
        gdown.download(gdrive_url, output_filename, quiet=False)
        df = pd.read_csv(output_filename)
        if show_head:
            display(df.head(3))
    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please ensure the file ID is correct and the file is publicly accessible.")
    finally:
        return df


# Data Mounting

In [4]:
Thefile = '/content/drive/My Drive/Colab Notebooks/ga4_obfuscated_sample_ecommerce.parquet'
Already_Parquet = True  if os.path.exists(Thefile) else False

if not Already_Parquet:
    file_id = '14dtPTtcGX_Hg-Pteg-_ovqbZX1eID24A'
    output_filename = 'ga4_obfuscated_sample_ecommerce.csv'
    Data = CSV_gdrive(file_id, output_filename, show_head=True)

Downloading...
From (original): https://drive.google.com/uc?id=14dtPTtcGX_Hg-Pteg-_ovqbZX1eID24A
From (redirected): https://drive.google.com/uc?id=14dtPTtcGX_Hg-Pteg-_ovqbZX1eID24A&confirm=t&uuid=448e3697-398c-40d4-a15a-677ff77a01ad
To: /content/ga4_obfuscated_sample_ecommerce.csv
100%|██████████| 365M/365M [00:02<00:00, 123MB/s] 


,user_id,product_id,product_name,department_id,aisle_id,brand,category3,avg_interaction_price,view_count,add_to_cart_count,...,item_total_units_sold,item_avg_price,last_interaction_raw_unix_micros,last_interaction_date_gmt9,first_interaction_date_gmt9,last_interaction_time_gmt9,first_interaction_time_gmt9,interaction_duration_hours,last_interaction_day_of_week_gmt9,last_interaction_hour_category_gmt9
0,6.662725e+07,GGOEGXXX1613,Super G Unisex Joggers,Home/New/,(not set),Google,(not set),37.0,4,0,...,0,36.981059,1609580429985271,2021-01-02,2020-12-01,18:40:29.985271,22:56:15.931913,763,Saturday,18
1,6.662725e+07,GGCOGBJD157199,Google Land & Sea Tote Bag,Home/New/,(not set),(not set),(not set),20.0,4,1,...,0,19.987555,1609580429985271,2021-01-02,2020-12-01,18:40:29.985271,22:56:15.931913,763,Saturday,18
2,6.662725e+07,GGCOGXXX1569,Google Land & Sea Unisex Tee,Home/New/,(not set),(not set),(not set),25.0,4,1,...,0,24.902469,1609580429985271,2021-01-02,2020-12-01,18:40:29.985271,22:56:15.931913,763,Saturday,18


In [5]:
if not Already_Parquet:
    dirLoc = '/content/drive/My Drive/Colab Notebooks/'
    output = dirLoc + 'ga4_obfuscated_sample_ecommerce.parquet'
    Data.to_parquet(output, compression='brotli')
    print(f"DataFrame saved to {output} with Brotli compression.")

DataFrame saved to ga4_obfuscated_sample_ecommerce.parquet with Brotli compression.


In [17]:
if Already_Parquet:
    MasterData = pd.read_parquet(Thefile)

# Data Engineering

## 0. Imports & Global Configuration

In [22]:
!pip install -q tqdm_joblib mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

In [23]:
from __future__ import annotations

import os
import logging
import warnings
from typing import Dict, List, Tuple, Optional, Union, Any

import numpy as np
import pandas as pd
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

from joblib import Parallel, delayed, dump, load
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

import mlflow
import cloudpickle

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s"
)

logger = logging.getLogger(__name__)

## 1. DuckDB Connection Manager

In [24]:
def create_duckdb_connection(
    db_path: Optional[str] = None
) -> duckdb.DuckDBPyConnection:

    logger.debug("Creating DuckDB connection")

    try:
        if db_path:
            conn = duckdb.connect(db_path)
        else:
            conn = duckdb.connect()

        logger.debug("DuckDB connection created")
        return conn

    except Exception as e:
        logger.exception("DuckDB connection failed")
        raise ValueError("DuckDB connection failed") from e

## 2. CSV --> Parquet Converter

In [25]:
def csv_to_parquet(
    csv_path: str,
    parquet_path: str
) -> None:

    logger.debug("Converting CSV to Parquet")

    if not os.path.exists(csv_path):
        raise ValueError("CSV file not found")

    try:
        duckdb.query(
            f"""
            COPY (
                SELECT * FROM read_csv_auto('{csv_path}')
            )
            TO '{parquet_path}' (FORMAT PARQUET)
            """
        )

    except Exception as e:
        logger.exception("CSV conversion failed")
        raise ValueError("CSV to parquet failed") from e

## 3. Load Parquet into DuckDB

In [26]:
def load_parquet_to_duckdb(
    conn: duckdb.DuckDBPyConnection,
    parquet_path: str,
    table_name: str
) -> None:

    logger.debug("Loading parquet to DuckDB")

    if not os.path.exists(parquet_path):
        raise ValueError("Parquet file not found")

    try:
        conn.execute(
            f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM read_parquet('{parquet_path}')"
        )

    except Exception as e:
        logger.exception("Load parquet failed")
        raise ValueError("Load parquet failed") from e

## 4. DuckDB Query Executor

In [27]:
def run_duckdb_query(
    conn: duckdb.DuckDBPyConnection,
    query: str,
    return_df: bool = True
) -> Optional[pd.DataFrame]:

    logger.debug("Executing DuckDB query")

    try:
        result = conn.execute(query)

        if return_df:
            return result.df()

        return None

    except Exception as e:
        logger.exception("DuckDB query failed")
        raise ValueError("DuckDB query failed") from e

## 5. DuckDB Missing Summary

In [28]:
def duckdb_missing_summary(
    conn: duckdb.DuckDBPyConnection,
    table_name: str
) -> pd.DataFrame:

    logger.debug("Computing missing summary")

    try:
        cols = conn.execute(
            f"PRAGMA table_info({table_name})"
        ).df()["name"].tolist()

        queries = [
            f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) as {c}"
            for c in cols
        ]

        query = f"SELECT {','.join(queries)} FROM {table_name}"

        return conn.execute(query).df()

    except Exception as e:
        logger.exception("Missing summary failed")
        raise ValueError("Missing summary failed") from e

## 6. DuckDB User Level Features

In [29]:
def create_user_level_features_duckdb(
    conn: duckdb.DuckDBPyConnection,
    table_name: str,
    user_col: str
) -> pd.DataFrame:

    logger.debug("Creating user level features")

    query = f"""
    SELECT
        {user_col},
        COUNT(*) as total_events,
        COUNT(DISTINCT session_id) as total_sessions,
        SUM(engagement_time_msec) as total_engagement,
        AVG(engagement_time_msec) as avg_engagement
    FROM {table_name}
    GROUP BY {user_col}
    """

    try:
        return conn.execute(query).df()

    except Exception as e:
        logger.exception("Feature aggregation failed")
        raise ValueError("Feature aggregation failed") from e

## 7. Safe Sampling for Visualization

In [30]:
def duckdb_sample(
    conn: duckdb.DuckDBPyConnection,
    table_name: str,
    fraction: float = 0.1
) -> pd.DataFrame:

    logger.debug("Sampling DuckDB table")

    query = f"SELECT * FROM {table_name} USING SAMPLE {fraction}"

    return conn.execute(query).df()

## 8. Label Encoding (Production Safe)

In [31]:
def fit_label_encoders(
    df: pd.DataFrame,
    columns: List[str],
    save_path: str
) -> Dict[str, LabelEncoder]:

    logger.debug("Fitting label encoders")

    encoders: Dict[str, LabelEncoder] = {}

    try:
        for col in columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le

        dump(encoders, save_path)

        return encoders

    except Exception as e:
        logger.exception("Label encoder failed")
        raise ValueError("Label encoder failed") from e

## 9. Reverse Label Encoding

In [32]:
def inverse_label_encoding(
    df: pd.DataFrame,
    encoder_path: str
) -> pd.DataFrame:

    logger.debug("Inverse label encoding")

    encoders = load(encoder_path)

    for col, encoder in encoders.items():
        df[col] = encoder.inverse_transform(df[col])

    return df

## 10. Visualization (Safe Sampling Only)

In [33]:
def plot_distribution(
    df: pd.DataFrame,
    column: str
) -> None:

    logger.debug("Plotting distribution")

    plt.figure(figsize=(10,6))
    sns.histplot(df[column], kde=True)
    plt.show()

## 11. MLflow Tracking

In [34]:
def start_mlflow_experiment(
    experiment_name: str
) -> None:

    logger.debug("Starting MLflow")

    mlflow.set_experiment(experiment_name)
    mlflow.start_run()

## 12. Genetic Feature Selection

In [35]:
def genetic_feature_selection(
    X: pd.DataFrame,
    y: pd.Series,
    generations: int = 5
) -> List[str]:

    logger.debug("Genetic feature selection")

    from sklearn.ensemble import RandomForestClassifier

    features = list(X.columns)
    best_features = features

    for _ in tqdm(range(generations), colour="green"):

        subset = np.random.choice(
            features,
            size=len(features)//2,
            replace=False
        )

        model = RandomForestClassifier()
        model.fit(X[list(subset)], y)

        score = model.score(X[list(subset)], y)

        if score > 0.5:
            best_features = subset

    return list(best_features)

## 13. Save Pipeline

In [36]:
def save_pipeline(
    pipeline: Any,
    path: str
) -> None:

    logger.debug("Saving pipeline")

    with open(path, "wb") as f:
        cloudpickle.dump(pipeline, f)

## 14. Load Pipeline

In [37]:
def load_pipeline(
    path: str
) -> Any:

    logger.debug("Loading pipeline")

    with open(path, "rb") as f:
        return cloudpickle.load(f)